In [ ]:
import pandas as pd

url = 'https://docs.google.com/spreadsheets/d/1rzqggGnZIEa_vO5GFTWHwkq_yvvOvkRKhuexB6Hz-OU/export?format=csv';

dataset = pd.read_csv(url);

# Link para o código do Colab: https://colab.research.google.com/drive/14qkbt-o2OjdU7wu3dUnKaRJOktvmMG0I?usp=sharing


In [ ]:
COLUMN_MAP = {
    '#': 'index',
    'Idade': 'idade',
    'Diagnóstico': 'diagnostico',
    'Astigmatismo': 'astigmatismo',
    'Taxa lacrimal': 'tx_lacrimal',
    'Tipo de lente': 'tipo_lente',
}

# 🔹 Renomear dataset
ds_maped = dataset.rename(columns=COLUMN_MAP)

In [ ]:
# Questão 1

# Supondo um classificador Naive Bayes, construa a tabela de contagens e probabilidades 
# suavizadas usando o estimador laplaciano (α=1) para conjunto Lentes de Contato.

import pandas as pd

alpha = 1

for col in ["idade", "diagnostico", "astigmatismo", "tx_lacrimal"]:
    print(f"\nFeature: {col}")
    
    tabela = pd.crosstab(ds_maped[col], ds_maped["tipo_lente"])
    
    # aplica Laplace
    probs = (tabela + alpha) / (tabela.sum(axis=0) + alpha * tabela.shape[0])
    
    print(probs)

In [ ]:
# Questão 2 

# Dado o paciente abaixo, mostre a previsão dada pelo classificador Naive Bayes.
# - Apresente as probabilidades normalizadas para as três classes possíveis.

#   +----------+----------------+--------------+---------------+---------------+
#   |  Idade   |  Diagnóstico   | Astigmatismo | Taxa lacrimal | Tipo de lente |
#   +----------+----------------+--------------+---------------+---------------+
#   | infantil | hipermetropia  |      não     |    reduzida   |       ?       |
#   +----------+----------------+--------------+---------------+---------------+

paciente = {
    "idade": "infantil",
    "diagnostico": "hipermetropia",
    "astigmatismo": "não",
    "tx_lacrimal": "reduzida"
}

classes = ds_maped["tipo_lente"].unique()
resultados = {}

for c in classes:
    prob = 1
    
    # prior
    prob *= (ds_maped["tipo_lente"].value_counts()[c] / len(ds_maped))
    
    for col, val in paciente.items():
        count = len(ds_maped[(ds_maped[col] == val) & (ds_maped["tipo_lente"] == c)])
        total = len(ds_maped[ds_maped["tipo_lente"] == c])
        k = ds_maped[col].nunique()
        
        prob *= (count + 1) / (total + k)  # Laplace
    
    resultados[c] = prob

# 🔹 Normalizar
soma = sum(resultados.values())
for c in resultados:
    resultados[c] /= soma

print("\nResultados: ")
print("\tNenhuma: ", resultados["nenhuma"])
print("\tGelatinosa: ", resultados["gelatinosa"])
print("\tDura: ", resultados["dura"])

print("Classe prevista:", max(resultados, key=resultados.get))

In [ ]:
# Questão 3

# Supondo que há uma relação de dependência entre os atributos Idade e Taxa Lacrimal (o 2º é dependente do 1º) e que todos são 
# dependentes da classe, construa a rede bayesiana correspondente (estrutura e tabelas de probabilidades condicionais).
# - Suavize as probabilidades usando o estimador laplaciano com α=1

alpha = 1

# P(tx_lacrimal | idade)
tabela = pd.crosstab(ds_maped["tx_lacrimal"], ds_maped["idade"])
probs = (tabela + alpha) / (tabela.sum(axis=0) + alpha * tabela.shape[0])
print(probs)

# P(tipo_lente | tudo)
tabela = pd.crosstab(
    [ds_maped["idade"], ds_maped["diagnostico"], ds_maped["astigmatismo"], ds_maped["tx_lacrimal"]],
    ds_maped["tipo_lente"]
)
probs = (tabela + alpha) / (tabela.sum(axis=1) + alpha * tabela.shape[1]).values.reshape(-1, 1)
print(probs)

In [ ]:
# Questão 4

# Reclassifique o paciente dado no exercício 2 usando a rede bayesiana do exercício 3.

tabela = pd.crosstab(
    [ds_maped["idade"], ds_maped["diagnostico"], ds_maped["astigmatismo"], ds_maped["tx_lacrimal"]],
    ds_maped["tipo_lente"]
)

probs = (tabela + alpha) / (tabela.sum(axis=1) + alpha * tabela.shape[1]).values.reshape(-1, 1)

print(probs)